In [ ]:
# =============================================
# MERCHANT CHURN & RETENTION ANALYSIS
# Author: Fadwa Ramadan Ali Hassan
# Date: May 2026
# =============================================

## Imports

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Load the data

In [5]:
# Load the data
df = pd.read_csv("C:\my files\projects\DA\Merchant_Churn_Retention\data\online_retail_II.csv")   # ← Change filename if different

print("✅ Dataset Loaded Successfully")
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nFirst 5 rows:")
print(df.head())
print("\nData Info:")
print(df.info())

✅ Dataset Loaded Successfully
Shape: (1067371, 8)

Columns:
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']

First 5 rows:
  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323P                   PINK CHERRY LIGHTS        12   
2  489434    79323W                  WHITE CHERRY LIGHTS        12   
3  489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   

           InvoiceDate  Price  Customer ID         Country  
0  2009-12-01 07:45:00   6.95      13085.0  United Kingdom  
1  2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
2  2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
3  2009-12-01 07:45:00   2.10      13085.0  United Kingdom  
4  2009-12-01 07:45:00   1.25      13085.0  United Kingdom  

Data Info:
<class 'pandas.core.fram

## Data Cleaning

In [6]:
df_clean = df.copy()

# Rename columns
df_clean = df_clean.rename(columns={
    'Customer ID': 'MerchantID',
    'InvoiceDate': 'TransactionDate'
})

In [ ]:
missing = pd.DataFrame({
    'Missing Count': df_clean.isnull().sum(),
    'Missing %': round(df_clean.isnull().sum() / len(df_clean) * 100, 2)
})
print("=== Missing Values Analysis ===")
print(missing)

print("\nDropping rows with missing MerchantID...")
df_clean = df_clean.dropna(subset=['MerchantID'])

df_clean = df_clean[~df_clean['Invoice'].astype(str).str.startswith('C')]

df_clean = df_clean[df_clean['Quantity'] > 0]

df_clean['TransactionDate'] = pd.to_datetime(df_clean['TransactionDate'])

print("\n✅ Cleaning Completed")
print("Original Shape:", df.shape)
print("Cleaned Shape:", df_clean.shape)
print("Final Churn-able Records:", len(df_clean))

=== Missing Values Analysis ===
                 Missing Count  Missing %
Invoice                      0       0.00
StockCode                    0       0.00
Description               4382       0.41
Quantity                     0       0.00
TransactionDate              0       0.00
Price                        0       0.00
MerchantID              243007      22.77
Country                      0       0.00

Dropping rows with missing MerchantID...

✅ Cleaning Completed
Original Shape: (1067371, 8)
Cleaned Shape: (805620, 8)
Final Churn-able Records: 805620


drop rows with missing MerchantID because Without MerchantID, cohort analysis or merchant-level tracking won't be doable.

## Merchant-Level Data + Cohorts

In [9]:
# Create a merchant-level summary
merchant_df = df_clean.groupby('MerchantID').agg(
    Total_Transactions=('Invoice', 'nunique'),
    Total_Quantity=('Quantity', 'sum'),
    Total_Spend=('Price', lambda x: (x * df_clean.loc[x.index, 'Quantity']).sum()),
    First_Transaction=('TransactionDate', 'min'),
    Last_Transaction=('TransactionDate', 'max'),
    Country=('Country', 'first'),                    # Most common country
    Avg_Order_Value=('Price', 'mean')
).reset_index()

# Create Cohort (Month of first transaction)
merchant_df['Cohort_Month'] = merchant_df['First_Transaction'].dt.to_period('M')

# Calculate Tenure (in months)
merchant_df['Tenure_Months'] = ((merchant_df['Last_Transaction'] - merchant_df['First_Transaction']).dt.days / 30).round(1)

print("✅ Merchant-Level Table Created")
print("Number of Unique Merchants:", merchant_df.shape[0])
print("\nFirst 5 Merchants:")
print(merchant_df.head())

print("\nCohort Distribution (Top 10):")
print(merchant_df['Cohort_Month'].value_counts().head(10))

✅ Merchant-Level Table Created
Number of Unique Merchants: 5881

First 5 Merchants:
   MerchantID  Total_Transactions  Total_Quantity  Total_Spend  \
0     12346.0                  12           74285     77556.46   
1     12347.0                   8            3286      5633.32   
2     12348.0                   5            2714      2019.40   
3     12349.0                   4            1624      4428.69   
4     12350.0                   1             197       334.40   

    First_Transaction    Last_Transaction         Country  Avg_Order_Value  \
0 2009-12-14 08:34:00 2011-01-18 10:01:00  United Kingdom         6.100000   
1 2010-10-31 14:20:00 2011-12-07 15:52:00         Iceland         2.546087   
2 2010-09-27 14:59:00 2011-09-25 13:13:00         Finland         3.786275   
3 2010-04-29 13:20:00 2011-11-21 09:51:00           Italy         8.459657   
4 2011-02-02 16:01:00 2011-02-02 16:01:00          Norway         3.841176   

  Cohort_Month  Tenure_Months  
0      2009-12    

## Define Churn + Retention Analysis

In [ ]:
# Define Churn Logic
# A merchant is considered churned if they have not made any transaction in the last 90 days
reference_date = df_clean['TransactionDate'].max()
print("Reference Date (Latest in dataset):", reference_date)

merchant_df['Days_Since_Last_Transaction'] = (reference_date - merchant_df['Last_Transaction']).dt.days

# Churn Definition: No transaction in last 90 days
merchant_df['Is_Churned'] = merchant_df['Days_Since_Last_Transaction'] > 90
merchant_df['Is_Active'] = ~merchant_df['Is_Churned']

print("\nChurn Rate:")
print(merchant_df['Is_Churned'].value_counts())
print(merchant_df['Is_Churned'].value_counts(normalize=True) * 100)

# Cohort Retention Analysis
cohort_retention = merchant_df.groupby('Cohort_Month').agg(
    Total_Merchants=('MerchantID', 'nunique'),
    Active_Merchants=('Is_Active', 'sum'),
    Churned_Merchants=('Is_Churned', 'sum'),
    Avg_Tenure=('Tenure_Months', 'mean')
).round(2)

cohort_retention['Retention_Rate_%'] = round(cohort_retention['Active_Merchants'] / cohort_retention['Total_Merchants'] * 100, 2)

print("\n=== Cohort Retention Summary ===")
print(cohort_retention.head(12))

Reference Date (Latest in dataset): 2011-12-09 12:50:00

Churn Rate:
Is_Churned
True     2987
False    2894
Name: count, dtype: int64
Is_Churned
True     50.790682
False    49.209318
Name: proportion, dtype: float64

=== Cohort Retention Summary ===
              Total_Merchants  Active_Merchants  Churned_Merchants  \
Cohort_Month                                                         
2009-12                   955               573                382   
2010-01                   383               159                224   
2010-02                   376               152                224   
2010-03                   443               179                264   
2010-04                   294               114                180   
2010-05                   254                82                172   
2010-06                   270                86                184   
2010-07                   186                68                118   
2010-08                   162                61   

## Data exportation

In [ ]:
# 1. Merchant Level Data (Main table)
merchant_df.to_csv(r"C:\my files\projects\DA\Merchant_Churn_Retention\data\merchant_level.csv", index=False)

# 2. Transaction Level (Detailed)
df_clean.to_csv(r"C:\my files\projects\DA\Merchant_Churn_Retention\data\transactions_clean.csv", index=False)

# 3. Cohort Retention Summary
cohort_retention.to_csv(r"C:\my files\projects\DA\Merchant_Churn_Retention\data\cohort_retention.csv")

# 4. Daily/Monthly Summary
daily_activity = df_clean.groupby(df_clean['TransactionDate'].dt.to_period('M')).agg(
    Total_Transactions=('Invoice', 'nunique'),
    Total_Merchants=('MerchantID', 'nunique'),
    Total_Spend=('Price', lambda x: (x * df_clean.loc[x.index, 'Quantity']).sum())
).round(2)

daily_activity.to_csv(r"C:\my files\projects\DA\Merchant_Churn_Retention\data\monthly_activity.csv")

print("✅ All Tables Saved Successfully for Power BI!")
print("- merchant_level.csv")
print("- transactions_clean.csv")
print("- cohort_retention.csv")
print("- monthly_activity.csv")

✅ All Tables Saved Successfully for Power BI!
- merchant_level.csv
- transactions_clean.csv
- cohort_retention.csv
- monthly_activity.csv
